In [1]:
import pandas as pd
url ="https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

2.1 Loading Data & First Look

In [2]:
df.head() #first five rows

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info() # column types, nulls, memory

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df.describe()    # statistical summary (mean, std, min, max...)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df.shape   # (rows, columns)

(891, 12)

In [6]:
df.columns  # list of column names

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

2.2 Selecting & Filtering (this is 80% of daily Pandas use)


In [7]:
# Select a column (returns a Series)
ages = df['Age']

# Select multiple columns (returns a DataFrame)
subset = df[['Name', 'Age', 'Survived']]

# Filter rows by condition
survivors = df[df['Survived'] == 1]
old_survivors = df[(df['Survived'] == 1) & (df['Age'] > 50)]   # AND
first_or_third = df[(df['Pclass'] == 1) | (df['Pclass'] == 3)]  # OR

In [8]:
# .loc (label-based) vs .iloc (position-based)
df.loc[0, 'Name']      # row 0, column 'Name'

'Braund, Mr. Owen Harris'

In [9]:
df.iloc[0, 3]           # row 0, 4th column (by position)

'Braund, Mr. Owen Harris'

2.3 Handling Missing Data (real datasets are always messy)


In [10]:
df.isnull().sum()              # count nulls per column

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [11]:
df['Age'].fillna(df['Age'].median(), inplace=True)   # fill with median
df.dropna(subset=['Embarked'], inplace=True)          # drop rows missing this col
df.drop('Cabin', axis=1, inplace=True)                 # drop a column entirely

2.4 GroupBy — Aggregating Data (this is where insights come from)


In [12]:
# Average survival rate by passenger class
df.groupby('Pclass')['Survived'].mean()


,Survived
Pclass,
1,0.626168
2,0.472826
3,0.242363


In [13]:
# Multiple aggregations at once
df.groupby('Sex').agg({
    'Age': 'mean',
    'Survived': 'sum',
    'Fare': 'max'
})

,Age,Survived,Fare
Sex,,,
female,27.788462,231,512.3292
male,30.140676,109,512.3292


In [14]:
# Sort results
df.groupby('Pclass')['Fare'].mean().sort_values(ascending=False)

,Fare
Pclass,
1,84.193516
2,20.662183
3,13.675550


2.5 Creating New Columns (feature engineering starts here)


In [15]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 60, 100], labels=['Child', 'Teen', 'Adult', 'Senior'])

This is literally called feature engineering — turning raw columns into more useful signals for a model. You'll do this in every project.


In [16]:
# 1. What % of passengers survived overall?
# 2. What was the average fare paid by survivors vs non-survivors?
# 3. Create a new column 'FamilySize' (SibSp + Parch + 1)
# 4. Group by 'Embarked' and show average survival rate per port

In [17]:
# 1.
survival_rate = df['Survived'].mean() * 100
print(f"{survival_rate:.1f}% survived")

38.2% survived


In [18]:
# 2.
df.groupby('Survived')['Fare'].mean()

,Fare
Survived,
0,22.117887
1,48.209498


In [19]:
# 3.
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [20]:
# 4.
df.groupby('Embarked')['Survived'].mean()

,Survived
Embarked,
C,0.553571
Q,0.389610
S,0.336957
